# ارزیابی واقعی Kronos در چند مبدأ تاریخی
فقط Google Colab. نوت‌بوک آماده است، نه اجراشده. نصب و وزن فقط در runtime ابری؛ بدون حساب MT5، توکن، ژورنال، سفارش یا سرور عمومی.
پیش‌فرض ثابت: ۵ مبدأ با حداقل ۶۰ دقیقه فاصله، افق‌های ۱۵/۳۰/۶۰، یک مبدأ final_test. حداقل ۴۲۰ کندل بستهٔ پیوستهٔ M1 لازم است؛ ۱۰۰۰ کندل پیشنهاد می‌شود.
نتیجهٔ خوب و بد هر دو دانلود می‌شوند. final_test برای تغییر مدل یا انتخاب نمونه استفاده نشود. پنج مبدأ برای اثبات سودآوری کافی نیست.


In [ ]:
# COLAB ONLY
import os, sys, subprocess, json, hashlib, time, math
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
assert IN_COLAB, 'This notebook must run only inside Google Colab runtime.'
print('Runtime Python:', sys.version.split()[0], '| executable:', sys.executable)
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError(
        'This pinned notebook supports Python 3.10-3.12 only. '
        'Python 3.13+ is not validated for these pinned dependencies. '
        'Select a Python 3.11/3.12 hosted runtime if available. '
        'Do not replace the system Python or silently upgrade dependency pins.')
KRONOS_COMMIT = '67b630e67f6a18c9e9be918d9b4337c960db1e9a'
WORK='/content/kronos-work'
REPO=f'{WORK}/Kronos'
os.makedirs(WORK, exist_ok=True)
# Install deps in Colab runtime only
# Install the pinned checkout's actual requirements after verification below.
if not os.path.exists(REPO):
    subprocess.check_call(['git','clone','https://github.com/shiyu-coder/Kronos.git',REPO])
subprocess.check_call(['git','-C',REPO,'fetch','--all'])
subprocess.check_call(['git','-C',REPO,'checkout',KRONOS_COMMIT])
head=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
dirty=subprocess.check_output(['git','-C',REPO,'status','--porcelain'],text=True).strip()
assert head==KRONOS_COMMIT and not dirty, 'Pinned clean Kronos checkout required'
# Record loaded distributions before pip; changing them requires a session restart.
from importlib import metadata as _metadata
_KRONOS_RUNTIME_READY = False
_watched = ('torch', 'numpy', 'pandas', 'matplotlib', 'einops', 'huggingface_hub', 'safetensors')
def _installed_version(name):
    try:
        return _metadata.version(name)
    except _metadata.PackageNotFoundError:
        return None
_before_versions = {name: _installed_version(name) for name in _watched}
_loaded_before_install = {name for name in _watched if name in sys.modules}
# Keep pip diagnostics visible; CalledProcessError alone hides the useful cause.
install = subprocess.run([sys.executable,'-m','pip','install','-r',os.path.join(REPO,'requirements.txt')])
if install.returncode:
    raise RuntimeError('Dependency installation failed. Read the pip ERROR lines above; do not run inference cells.')
_changed_loaded = [name for name in _loaded_before_install
                   if _before_versions[name] != _installed_version(name)]
if _changed_loaded:
    raise RuntimeError(
        'Installed versions of loaded packages changed: ' + ', '.join(sorted(_changed_loaded)) +
        '. Use Runtime > Restart session, then rerun setup and validation. '
        'Do not delete the runtime or reinstall torch manually.')
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Kronos checkout OK:', head)


In [ ]:
# COLAB ONLY - validate runtime BEFORE uploading data/loading Kronos
_KRONOS_RUNTIME_READY = False
try:
    import torch
    # Exercise a real operation, not just version metadata.
    assert torch.tensor([1.0, 2.0]).sum().item() == 3.0
    from model import Kronos, KronosPredictor, KronosTokenizer
except Exception as exc:
    raise RuntimeError(
        'PyTorch/Kronos runtime validation failed (' + type(exc).__name__ + '). '
        'If the output mentions TORCH_LIBRARY/prims or a partially initialized module, '
        'use Runtime > Restart session, then run setup and this validation cell again. '
        'Do not use importlib.reload(torch), delete sys.modules, or rerun inference. '
        'If it still fails in a fresh session, retain the full error above for diagnosis.'
    ) from exc
_KRONOS_RUNTIME_READY = True
print('Runtime validation OK | torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('Kronos import OK. You may now upload OHLCV data and run inference.')


## کد ارزیابی پین‌شده
از همان تابع تست‌شدهٔ پروژه استفاده می‌شود، نه کپی مستقل فرمول‌ها. کد GitHub در Colab دریافت و اجرا می‌شود. نسخه پیش از مشاهده نتیجه ثابت شده است.


In [ ]:
assert globals().get('_KRONOS_RUNTIME_READY') is True, 'Complete setup/validation first'
EVALUATOR_COMMIT='4f66fe889c615cb4b6d689f57d3d8ee1cda9ba3e'
EVALUATOR_REPO='/content/trade-mcp-evaluator'
if not os.path.exists(EVALUATOR_REPO):
    subprocess.check_call(['git','clone','https://github.com/enigmafarsco-lang/trade-mcp.git',EVALUATOR_REPO])
subprocess.check_call(['git','-C',EVALUATOR_REPO,'fetch','origin',EVALUATOR_COMMIT])
subprocess.check_call(['git','-C',EVALUATOR_REPO,'checkout',EVALUATOR_COMMIT])
assert subprocess.check_output(['git','-C',EVALUATOR_REPO,'rev-parse','HEAD'],text=True).strip()==EVALUATOR_COMMIT
assert not subprocess.check_output(['git','-C',EVALUATOR_REPO,'status','--porcelain'],text=True).strip()
sys.path.insert(0,os.path.join(EVALUATOR_REPO,'src'))
from trade_mcp.engines.upstream import minute_rows, fixed_multi_origin_splits, evaluate_kronos_multi_origin


## دادهٔ ورودی و طرح ثابت آزمون
فقط JSON بازار بدون اطلاعات حساب. شکاف با کندل ساختگی پر نمی‌شود؛ هنگام کمبود داده توقف می‌شود.


In [ ]:
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1, 'Upload exactly one redacted JSON file'
input_bytes=next(iter(uploaded.values()))
assert len(input_bytes)<=4_000_000, 'Input too large'
request=json.loads(input_bytes.decode('utf-8'))
assert set(request)<= {'symbol','rows','timeframe','timezone','selection'}, 'Unexpected input fields'
assert request.get('timeframe','M1')=='M1' and request.get('timezone','UTC')=='UTC'
symbol=str(request['symbol'])
assert symbol and len(symbol)<=64 and all(ch.isalnum() or ch in '._-' for ch in symbol)
rows=minute_rows(request['rows'])
assert 420<=len(rows)<=5000, 'Need 420..5000 contiguous closed M1 candles'
assert all(r[0]%60==0 and r[0]+60<=time.time() for r in rows), 'Closed UTC M1 only'
SETTINGS={'origins':5,'final_test_origins':1,'step':60,'lookback':15}
plan=fixed_multi_origin_splits(rows, origins=5, final_test_origins=1,step=60)
print('Rows:',len(rows),'| planned inference calls:',len(plan))
print('Fixed split labels:',[p['split'] for p in plan])
print('Do not adjust settings based on final_test results.')


## اجرای واقعی مدل
این سلول پنج بار inference انجام می‌دهد؛ هیچ retry یا جست‌وجوی نمونهٔ برنده ندارد. فقط پس از آماده‌شدن runtime اجرا شود.


In [ ]:
from model import Kronos,KronosTokenizer,KronosPredictor
import random
import numpy as np
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
TOKENIZER_ID='NeoQuasar/Kronos-Tokenizer-base'
MODEL_ID='NeoQuasar/Kronos-small'
tokenizer=KronosTokenizer.from_pretrained(TOKENIZER_ID)
model=Kronos.from_pretrained(MODEL_ID)
predictor=KronosPredictor(model,tokenizer)
report=evaluate_kronos_multi_origin(rows,predictor,**SETTINGS)
report.update(research_only=True,execution_authorized=False,validated_for_trading=False,
    evaluator_commit=EVALUATOR_COMMIT,upstream_commit=KRONOS_COMMIT,
    input_sha256=hashlib.sha256(input_bytes).hexdigest(),model_id=MODEL_ID,
    tokenizer_id=TOKENIZER_ID,seed=SEED,settings=SETTINGS,
    reproducibility_warning='Seeds do not guarantee GPU determinism; model IDs are not immutable weight revisions')
print('Evaluation split:',json.dumps(report['split_summary'].get('evaluation',[]),indent=2))
print('FINAL TEST (not used for tuning):',json.dumps(report['split_summary'].get('final_test',[]),indent=2))
print('Postprocessing:',json.dumps(report['postprocessing_summary'],indent=2))


## دانلود گزارش، مستقل از خوب یا بد بودن نتیجه
این فایل گزارش تحقیقاتی است؛ artifact پیش‌بینی زنده نیست و نباید به مسیر چارت زنده یا سفارش متصل شود.


In [ ]:
output='/content/kronos_multi_origin_research.json'
with open(output,'w',encoding='utf-8') as stream:
    json.dump(report,stream,ensure_ascii=False,indent=2,allow_nan=False)
files.download(output)
print('Research report exported. No live forecast or trading authorization.')
